[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/057.%20The%20Periodic%20Review%20%28Base-Stock%29%20Policy%20Problem/P57-Tier-3_executed.ipynb) [![Open In SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open In Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 57. The Periodic Review (Base-Stock) Policy Problem

## Tier 3: A Genetic Algorithm for Multi-Objective Optimization

### Purpose and scope

This notebook applies the Non-dominated Sorting Genetic Algorithm II (NSGA-II) to a periodic-review base-stock policy. It searches for a set of policies that represent trade-offs among operating cost, service level, inventory investment, and ordering frequency. A decision maker can then compare these policies using strategic preferences.

The decision variables are the review period, $R$ (weeks), and the base-stock level, $S$ (units). The inputs are mean demand, demand standard deviation, lead time, holding cost, ordering cost, stockout cost, and unit inventory value. The outputs are an approximate Pareto front, a record of population performance by generation, decision-support comparisons, and visualizations.

### Model assumptions

Demand over a protection interval is normally distributed. The protection interval is the review period plus lead time, $R+L$. Demand has a constant mean rate $\mu$ and standard deviation rate $\sigma$, so the mean and standard deviation over an interval of length $t$ are $\mu t$ and $\sigma\sqrt{t}$. The model treats costs and demand parameters as constant and does not represent order quantities, capacity limits, quantity discounts, or time-varying demand. The service measure is the cycle service level: the probability that demand during a protection interval does not exceed the base-stock level.

### Policy model and objectives

For a policy $(R,S)$, define the protection interval and safety stock as

$$
T = R+L, \qquad SS = S-\mu T.
$$

Here, $T$ is measured in weeks, and $SS$ is the base-stock level minus expected demand over the protection interval. The normal-demand standard deviation over this interval is $\sigma_T=\sigma\sqrt{T}$. The standardized safety stock is $z=SS/\sigma_T$. The cycle service level is $\Phi(z)$, where $\Phi$ is the standard normal cumulative distribution function.

The expected shortage in one protection interval is

$$
E[Q] = \sigma_T\left[\phi(z)-z\left(1-\Phi(z)\right)\right],
$$

where $\phi$ is the standard normal probability density function. This expression applies whether $SS$ is positive or negative. In particular, when $S$ is below expected protection-interval demand, $z$ is negative; the service level is still $\Phi(z)$, and the expected shortage remains defined by the same normal-loss expression.

The cost and inventory calculations used in this model are

$$
\begin{aligned}
C_{\mathrm{order}} &= \frac{K}{R}, \\
I_{\mathrm{cycle}} &= \frac{\mu R}{2}, \\
I_{\mathrm{safety}} &= \max(0,SS), \\
I_{\mathrm{avg}} &= I_{\mathrm{cycle}}+I_{\mathrm{safety}}, \\
C_{\mathrm{hold}} &= hI_{\mathrm{avg}}, \\
C_{\mathrm{shortage}} &= \frac{p}{R}E[Q], \\
C_{\mathrm{total}} &= C_{\mathrm{order}}+C_{\mathrm{hold}}+C_{\mathrm{shortage}}, \\
V_{\mathrm{inventory}} &= vI_{\mathrm{avg}}, \\
f_{\mathrm{order}} &= \frac{1}{R}.
\end{aligned}
$$

Here, $K$ is the cost per order, $h$ is the holding cost per unit per week, $p$ is the shortage cost per unit, and $v$ is the value per inventory unit. The model charges expected shortage cost per review period and converts it to a weekly rate by dividing by $R$. The average-inventory expression is the cycle-stock term plus nonnegative safety stock, as specified by this model.

All four objectives are expressed as minimization objectives. Service is therefore negated in the objective vector:

$$
\min_{R,S}
\left(
C_{\mathrm{total}},
-\Phi(z),
V_{\mathrm{inventory}},
f_{\mathrm{order}}
\right),
\qquad
R\in[0.1,5.0],\quad S\in[1000,50000].
$$

The first objective minimizes weekly total cost; the second maximizes cycle service level by minimizing its negative; the third minimizes inventory investment; and the fourth minimizes ordering frequency, which favors less frequent ordering. No single policy is necessarily best on all four objectives.

### How NSGA-II searches

A **population** is a collection of candidate policies. Each candidate is an individual $(R,S)$. **Pareto dominance** compares candidates without combining objectives into one score: a candidate dominates another if it is no worse in every minimization objective and strictly better in at least one. The **Pareto front** is the set of candidates in the evaluated population that are not dominated by another candidate.

NSGA-II repeatedly ranks candidates into non-dominated fronts. It preserves the best fronts first; if only part of a front fits in the next population, **crowding distance** favors candidates in less crowded regions of objective space. This helps retain a spread of trade-off policies. Tournament selection chooses parents using front rank and crowding distance; simulated binary crossover and polynomial mutation generate new real-valued policies within the decision bounds. Parents and offspring are then considered together for the next population, giving the search its elitist selection step.

For population size $N$, objective count $M$, and $G$ generations, pairwise non-dominated sorting takes approximately $O(MN^2)$ work per generation; crowding-distance calculations require sorting objective values and take approximately $O(MN\log N)$. The algorithm is stochastic and returns an approximation based on the finite population and generation count; it does not guarantee the global Pareto front.

### Reading the results

The cost, service, investment, and ordering-frequency values in the output are evaluated for the same candidate policy. A Pareto-front solution is efficient relative to the other evaluated candidates, not necessarily globally optimal. The weighted decision scenarios later in the notebook are decision-support rules applied to the approximated front; they do not change the NSGA-II search or make the selected policy universally best. Because the search is stochastic, the particular front can vary between runs.



In [ ]:
import random
import warnings
from math import sqrt

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import norm

warnings.filterwarnings("ignore")

# Consistent plot styling keeps the multi-panel comparisons readable.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class MultiObjectiveBaseStockGA:
    """Apply NSGA-II to a multi-objective periodic-review base-stock policy."""

    def __init__(
        self,
        demand_mean,
        demand_std,
        lead_time,
        holding_cost,
        ordering_cost,
        stockout_cost,
        unit_value,
    ):
        """Store the model inputs and genetic algorithm settings."""
        self.mu = demand_mean
        self.sigma = demand_std
        self.L = lead_time
        self.h = holding_cost
        self.K = ordering_cost
        self.p = stockout_cost
        self.unit_value = unit_value

        self.pop_size = 15
        self.generations = 25
        self.cx_rate = 0.8
        self.mut_rate = 0.1
        self.tour_size = 3

        self.R_bounds = (0.1, 5.0)
        self.S_bounds = (1000, 50000)
        self.convergence_history = []

        if self.pop_size < 1:
            raise ValueError("Population size must be positive.")
        if self.tour_size < 1:
            raise ValueError("Tournament size must be positive.")
        if self.generations < 1:
            raise ValueError("The number of generations must be positive.")
        if demand_std <= 0 or demand_mean < 0 or lead_time < 0:
            raise ValueError("Demand standard deviation must be positive; demand mean and lead time must be nonnegative.")

        print("Initialized Multi-Objective GA:")
        print(f"  Population: {self.pop_size}, Generations: {self.generations}")
        print("  Objectives: Cost, Service, Inventory, Ordering Frequency")
        print(f"  Decision Variables: R in {self.R_bounds}, S in {self.S_bounds}")

    def create_individual(self):
        """Sample a review period and base-stock level within their bounds."""
        review_period = random.uniform(*self.R_bounds)
        base_stock_level = random.uniform(*self.S_bounds)
        return (review_period, base_stock_level)

    def evaluate_objectives(self, individual):
        """Return cost, negative service, investment, and order frequency."""
        review_period, base_stock_level = individual
        protection_interval = review_period + self.L

        ordering_cost = self.K / review_period
        cycle_stock = self.mu * review_period / 2
        safety_stock = base_stock_level - self.mu * protection_interval
        average_inventory = cycle_stock + max(0.0, safety_stock)
        holding_cost = self.h * average_inventory

        demand_std_interval = self.sigma * sqrt(protection_interval)
        z = safety_stock / demand_std_interval
        stockout_probability = norm.sf(z)
        expected_shortage = demand_std_interval * (
            norm.pdf(z) - z * stockout_probability
        )
        stockout_cost = (self.p / review_period) * expected_shortage
        total_cost = ordering_cost + holding_cost + stockout_cost

        service_level = norm.cdf(z)
        inventory_investment = average_inventory * self.unit_value
        ordering_frequency = 1 / review_period

        return [
            total_cost,
            -service_level,
            inventory_investment,
            ordering_frequency,
        ]

    @staticmethod
    def dominates(obj1, obj2):
        """Return whether obj1 is no worse in every objective and better in one."""
        no_worse_in_all = all(value1 <= value2 for value1, value2 in zip(obj1, obj2))
        strictly_better_in_one = any(
            value1 < value2 for value1, value2 in zip(obj1, obj2)
        )
        return no_worse_in_all and strictly_better_in_one

    def non_dominated_sort(self, objectives):
        """Partition candidates into successive non-dominated fronts."""
        n = len(objectives)
        dominated = [[] for _ in range(n)]
        domination_count = [0] * n
        first_front = []

        for i in range(n):
            for j in range(i + 1, n):
                if self.dominates(objectives[i], objectives[j]):
                    dominated[i].append(j)
                    domination_count[j] += 1
                elif self.dominates(objectives[j], objectives[i]):
                    dominated[j].append(i)
                    domination_count[i] += 1

            if domination_count[i] == 0:
                first_front.append(i)

        fronts = [first_front]
        front_index = 0
        while front_index < len(fronts) and fronts[front_index]:
            next_front = []
            for i in fronts[front_index]:
                for j in dominated[i]:
                    domination_count[j] -= 1
                    if domination_count[j] == 0:
                        next_front.append(j)
            if next_front:
                fronts.append(next_front)
            front_index += 1

        return fronts

    def calculate_crowding_distance(self, objectives):
        """Measure objective-space isolation within one non-dominated front."""
        n = len(objectives)
        distances = [0.0] * n
        if n <= 2:
            return [float("inf")] * n

        num_objectives = len(objectives[0])
        for objective_index in range(num_objectives):
            sorted_indices = sorted(
                range(n), key=lambda i: objectives[i][objective_index]
            )
            distances[sorted_indices[0]] = float("inf")
            distances[sorted_indices[-1]] = float("inf")

            objective_range = (
                objectives[sorted_indices[-1]][objective_index]
                - objectives[sorted_indices[0]][objective_index]
            )
            if objective_range == 0:
                continue

            for position in range(1, n - 1):
                current_index = sorted_indices[position]
                previous_value = objectives[sorted_indices[position - 1]][objective_index]
                next_value = objectives[sorted_indices[position + 1]][objective_index]
                distances[current_index] += (
                    next_value - previous_value
                ) / objective_range

        return distances

    def nsga2_selection(self, population, objectives):
        """Keep the best-ranked candidates, using crowding to split a front."""
        fronts = self.non_dominated_sort(objectives)
        selected_indices = []

        for front in fronts:
            available = self.pop_size - len(selected_indices)
            if len(front) <= available:
                selected_indices.extend(front)
            else:
                front_objectives = [objectives[i] for i in front]
                distances = self.calculate_crowding_distance(front_objectives)
                ranked_positions = sorted(
                    range(len(front)),
                    key=lambda position: distances[position],
                    reverse=True,
                )
                selected_indices.extend(
                    front[position] for position in ranked_positions[:available]
                )
                break

        return [population[i] for i in selected_indices]

    def tournament_selection(self, population, objectives):
        """Choose a parent by non-dominated rank, then by crowding distance."""
        sample_size = min(self.tour_size, len(population))
        sampled_indices = random.sample(range(len(population)), sample_size)
        fronts = self.non_dominated_sort(objectives)

        ranks = {}
        crowding = {}
        for rank, front in enumerate(fronts):
            distances = self.calculate_crowding_distance(
                [objectives[i] for i in front]
            )
            for position, index in enumerate(front):
                ranks[index] = rank
                crowding[index] = distances[position]

        winner = min(
            sampled_indices,
            key=lambda index: (ranks[index], -crowding[index]),
        )
        return population[winner]

    def crossover(self, parent1, parent2):
        """Apply simulated binary crossover to the two real-valued decisions."""
        if random.random() > self.cx_rate:
            return parent1, parent2

        offspring1 = []
        offspring2 = []
        bounds = [self.R_bounds, self.S_bounds]
        eta_c = 20

        for gene_index in range(2):
            p1 = parent1[gene_index]
            p2 = parent2[gene_index]
            lower_bound, upper_bound = bounds[gene_index]

            if abs(p1 - p2) > 1e-6:
                lower_parent = min(p1, p2)
                upper_parent = max(p1, p2)
                random_value = random.random()

                if random_value <= 0.5:
                    beta_q = (2 * random_value) ** (1 / (eta_c + 1))
                else:
                    beta_q = (1 / (2 * (1 - random_value))) ** (
                        1 / (eta_c + 1)
                    )

                child1 = 0.5 * (
                    (lower_parent + upper_parent)
                    - beta_q * (upper_parent - lower_parent)
                )
                child2 = 0.5 * (
                    (lower_parent + upper_parent)
                    + beta_q * (upper_parent - lower_parent)
                )
                offspring1.append(
                    max(lower_bound, min(upper_bound, child1))
                )
                offspring2.append(
                    max(lower_bound, min(upper_bound, child2))
                )
            else:
                offspring1.append(p1)
                offspring2.append(p2)

        return tuple(offspring1), tuple(offspring2)

    def mutate(self, individual):
        """Apply polynomial mutation while keeping genes within their bounds."""
        if random.random() > self.mut_rate:
            return individual

        mutated = []
        bounds = [self.R_bounds, self.S_bounds]
        eta_m = 20

        for gene_index in range(2):
            value = individual[gene_index]
            lower_bound, upper_bound = bounds[gene_index]
            delta1 = (value - lower_bound) / (upper_bound - lower_bound)
            delta2 = (upper_bound - value) / (upper_bound - lower_bound)
            random_value = random.random()

            if random_value <= 0.5:
                xy = 1 - delta1
                delta_q = (
                    2 * random_value
                    + (1 - 2 * random_value) * xy ** (eta_m + 1)
                ) ** (1 / (eta_m + 1)) - 1
            else:
                xy = 1 - delta2
                delta_q = 1 - (
                    2 * (1 - random_value)
                    + 2 * (random_value - 0.5) * xy ** (eta_m + 1)
                ) ** (1 / (eta_m + 1))

            new_value = value + delta_q * (upper_bound - lower_bound)
            mutated.append(max(lower_bound, min(upper_bound, new_value)))

        return tuple(mutated)

    def optimize(self):
        """Run the evolutionary search and return the final approximate front."""
        print("\nStarting NSGA-II optimization...")
        self.convergence_history = []
        population = [
            self.create_individual() for _ in range(self.pop_size)
        ]

        for generation in range(self.generations):
            objectives = [
                self.evaluate_objectives(individual)
                for individual in population
            ]
            services = [-objective[1] for objective in objectives]
            self.convergence_history.append(
                {
                    "generation": generation,
                    "best_cost": min(objective[0] for objective in objectives),
                    "best_service": max(services),
                    "avg_cost": float(np.mean([objective[0] for objective in objectives])),
                    "avg_service": float(np.mean(services)),
                }
            )

            if generation % 5 == 0:
                best_cost = self.convergence_history[-1]["best_cost"]
                best_service = self.convergence_history[-1]["best_service"]
                print(
                    f"Generation {generation}: Best Cost = ${best_cost:.2f}/week, "
                    f"Best Service = {best_service:.4f}"
                )

            offspring = []
            while len(offspring) < self.pop_size:
                parent1 = self.tournament_selection(population, objectives)
                parent2 = self.tournament_selection(population, objectives)
                child1, child2 = self.crossover(parent1, parent2)
                offspring.extend(
                    [self.mutate(child1), self.mutate(child2)]
                )

            offspring = offspring[: self.pop_size]
            combined_population = population + offspring
            combined_objectives = [
                self.evaluate_objectives(individual)
                for individual in combined_population
            ]
            population = self.nsga2_selection(
                combined_population, combined_objectives
            )

        final_objectives = [
            self.evaluate_objectives(individual) for individual in population
        ]
        first_front = self.non_dominated_sort(final_objectives)[0]
        pareto_front = []

        for index in first_front:
            individual = population[index]
            objective = final_objectives[index]
            pareto_front.append(
                {
                    "individual": individual,
                    "objectives": objective,
                    "review_period": individual[0],
                    "base_stock_level": individual[1],
                    "total_cost": objective[0],
                    "service_level": -objective[1],
                    "inventory_investment": objective[2],
                    "ordering_frequency": objective[3],
                }
            )

        return {
            "pareto_front": pareto_front,
            "convergence_history": self.convergence_history,
        }

print("MultiObjectiveBaseStockGA class defined successfully.")

In [ ]:
# Initialize the optimizer with the example's demand, cost, and lead-time inputs.
# Monetary cost parameters are interpreted in the units stated in the model.
ga_optimizer = MultiObjectiveBaseStockGA(
    demand_mean=12000,      # Mean demand per week, in units
    demand_std=2400,        # Demand standard deviation per square root week, in units
    lead_time=0.5,          # Lead time, in weeks
    holding_cost=0.15,      # Holding cost, per unit per week
    ordering_cost=75,       # Cost per order
    stockout_cost=8,        # Shortage cost per unit
    unit_value=5.0,         # Inventory value per unit
)

In [ ]:
ga_result = ga_optimizer.optimize()
pareto_front = ga_result["pareto_front"]
if not pareto_front:
    raise RuntimeError("The search returned no non-dominated solutions.")

pareto_sorted = sorted(pareto_front, key=lambda solution: solution["total_cost"])
pareto_df = pd.DataFrame(pareto_sorted)

print("\nNSGA-II OPTIMIZATION RESULTS")
print("=" * 60)
print(f"Approximate Pareto-front size: {len(pareto_df)} solutions")
print("\nRepresentative solutions from the evaluated front:")

representatives = [
    ("Lowest-cost", pareto_df.loc[pareto_df["total_cost"].idxmin()]),
    ("Median by cost", pareto_df.iloc[len(pareto_df) // 2]),
    ("Highest-service", pareto_df.loc[pareto_df["service_level"].idxmax()]),
]

for label, solution in representatives:
    print(f"\n{label} solution:")
    print(f"  Review period: {solution['review_period']:.3f} weeks")
    print(f"  Base-stock level: {solution['base_stock_level']:,.0f} units")
    print(f"  Total cost: ${solution['total_cost']:,.2f}/week")
    print(f"  Cycle service level: {solution['service_level']:.4f} ({solution['service_level'] * 100:.2f}%)")
    print(f"  Inventory investment: ${solution['inventory_investment']:,.2f}")
    print(f"  Ordering frequency: {solution['ordering_frequency']:.4f}/week")

costs = pareto_df["total_cost"].to_numpy()
services = pareto_df["service_level"].to_numpy()
investments = pareto_df["inventory_investment"].to_numpy()

print("\nObserved ranges on the approximate front:")
print(f"  Weekly cost: ${costs.min():,.2f} to ${costs.max():,.2f}")
print(f"  Cycle service level: {services.min():.4f} to {services.max():.4f}")
print(f"  Inventory investment: ${investments.min():,.2f} to ${investments.max():,.2f}")
if len(pareto_df) > 1 and np.ptp(costs) > 0 and np.ptp(services) > 0:
    correlation = np.corrcoef(costs, services)[0, 1]
    print(f"  Cost-service correlation among front candidates: {correlation:.3f}")
else:
    print("  Cost-service correlation is undefined for this front.")

In [ ]:
display_columns = [
    "review_period",
    "base_stock_level",
    "total_cost",
    "service_level",
    "inventory_investment",
    "ordering_frequency",
]

print("PARETO-FRONT CANDIDATES")
print("=" * 80)
print("\nLowest-cost candidates:")
print(
    pareto_df.nsmallest(10, "total_cost")[display_columns]
    .round(4)
    .to_string(index=False)
)

print("\nHighest-service candidates:")
print(
    pareto_df.nlargest(10, "service_level")[display_columns]
    .round(4)
    .to_string(index=False)
)

print("\nLowest-investment candidates:")
print(
    pareto_df.nsmallest(10, "inventory_investment")[display_columns]
    .round(4)
    .to_string(index=False)
)

# Efficiency ratios are descriptive comparisons, not additional optimization objectives.
service_percent = pareto_df["service_level"] * 100
positive_service = service_percent > 0
if positive_service.any():
    cost_per_service_point = (
        pareto_df.loc[positive_service, "total_cost"]
        / service_percent.loc[positive_service]
    )
    cost_efficient = pareto_df.loc[cost_per_service_point.idxmin()]
    print("\nLowest cost per percentage point of service among positive-service candidates:")
    print(
        f"  R={cost_efficient['review_period']:.3f} weeks, "
        f"S={cost_efficient['base_stock_level']:,.0f} units"
    )
    print(f"  Cost: ${cost_efficient['total_cost']:,.2f}/week")
else:
    print("\nCost per service point is undefined because all reported service levels are zero.")

simplicity_score = (
    pareto_df["service_level"] / pareto_df["ordering_frequency"]
)
simplest = pareto_df.loc[simplicity_score.idxmax()]
print("\nHighest service-to-ordering-frequency ratio among front candidates:")
print(
    f"  R={simplest['review_period']:.3f} weeks, "
    f"S={simplest['base_stock_level']:,.0f} units"
)
print(f"  Ordering frequency: {simplest['ordering_frequency']:.4f}/week")
print(f"  Cycle service level: {simplest['service_level']:.4f}")

In [ ]:
conv_df = pd.DataFrame(ga_result["convergence_history"])
if conv_df.empty:
    raise RuntimeError("No convergence history is available.")

initial = conv_df.iloc[0]
final = conv_df.iloc[-1]

print("GENERATION HISTORY")
print("=" * 50)
print(
    f"Best observed cost: ${initial['best_cost']:,.2f} to "
    f"${final['best_cost']:,.2f}/week"
)
print(
    f"Best observed cycle service: {initial['best_service']:.4f} to "
    f"{final['best_service']:.4f}"
)

cost_mean = conv_df["best_cost"].mean()
service_mean = conv_df["best_service"].mean()
cost_cv = conv_df["best_cost"].std(ddof=0) / cost_mean if cost_mean else np.nan
service_cv = (
    conv_df["best_service"].std(ddof=0) / service_mean
    if service_mean
    else np.nan
)
print(f"\nBest-cost coefficient of variation across recorded generations: {cost_cv:.4f}")
print(f"Best-service coefficient of variation across recorded generations: {service_cv:.4f}")

# These thresholds describe when a recorded best value first approaches its final value;
# they are not proof that the full multi-objective front has converged.
final_cost = final["best_cost"]
cost_threshold = final_cost * 1.01
cost_generation = int(
    conv_df.index[conv_df["best_cost"] <= cost_threshold][0]
)
final_service = final["best_service"]
service_threshold = final_service * 0.99
service_generation = int(
    conv_df.index[conv_df["best_service"] >= service_threshold][0]
)
print("\nFirst recorded generation within 1% of the final best value:")
print(f"  Cost: generation {cost_generation}")
print(f"  Service: generation {service_generation}")
print("These single-objective traces do not establish convergence of the entire Pareto front.")

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Multi-Objective Base-Stock Policy Search", fontsize=16, fontweight="bold")

# Color encodes review period, making each plotted policy interpretable in decision space.
scatter = ax1.scatter(
    pareto_df["total_cost"],
    pareto_df["service_level"] * 100,
    c=pareto_df["review_period"],
    cmap="viridis",
    s=55,
    alpha=0.8,
)
ax1.set_xlabel("Total cost ($/week)")
ax1.set_ylabel("Cycle service level (%)")
ax1.set_title("Approximate Pareto Front: Cost and Service")
ax1.grid(True, alpha=0.3)
fig.colorbar(scatter, ax=ax1, label="Review period (weeks)")

ax2.plot(
    conv_df["generation"],
    conv_df["best_cost"],
    color="tab:red",
    label="Best cost",
    linewidth=2,
)
ax2.plot(
    conv_df["generation"],
    conv_df["avg_cost"],
    color="tab:orange",
    linestyle="--",
    label="Mean population cost",
    linewidth=1.5,
)
ax2.set_xlabel("Generation")
ax2.set_ylabel("Cost ($/week)")
ax2.set_title("Cost Across Generations")
ax2.legend()
ax2.grid(True, alpha=0.3)

# Normalize each displayed measure to its observed range; service is shown as a
# performance measure, so a larger normalized service value means better service.
def normalize_series(values):
    values = np.asarray(values, dtype=float)
    value_range = np.ptp(values)
    if value_range == 0:
        return np.zeros_like(values)
    return (values - values.min()) / value_range

x_positions = np.arange(len(pareto_df))
bar_width = 0.25
normalized_cost = normalize_series(pareto_df["total_cost"])
normalized_service = normalize_series(pareto_df["service_level"])
normalized_investment = normalize_series(pareto_df["inventory_investment"])
ax3.bar(x_positions - bar_width, normalized_cost, bar_width, label="Cost (lower is better)")
ax3.bar(x_positions, normalized_service, bar_width, label="Service (higher is better)")
ax3.bar(
    x_positions + bar_width,
    normalized_investment,
    bar_width,
    label="Investment (lower is better)",
)
ax3.set_xlabel("Pareto-front candidate index (cost-sorted)")
ax3.set_ylabel("Normalized value (0–1)")
ax3.set_title("Relative Objective Values")
ax3.legend()
ax3.grid(True, axis="y", alpha=0.3)

space_scatter = ax4.scatter(
    pareto_df["review_period"],
    pareto_df["base_stock_level"],
    c=pareto_df["total_cost"],
    cmap="plasma",
    s=55,
    alpha=0.8,
)
ax4.set_xlabel("Review period (weeks)")
ax4.set_ylabel("Base-stock level (units)")
ax4.set_title("Decision Space: Review Period and Base-Stock Level")
ax4.grid(True, alpha=0.3)
fig.colorbar(space_scatter, ax=ax4, label="Total cost ($/week)")

fig.tight_layout()
plt.show()

print("The plots summarize the evaluated approximate front and the recorded generation history.")

In [ ]:
# These preference weights rank policies already found by NSGA-II; they do not
# alter the genetic algorithm or add a fifth objective to its Pareto search.
scenarios = {
    "Cost-conscious": {
        "weight_cost": 0.7,
        "weight_service": 0.2,
        "weight_investment": 0.1,
    },
    "Service-focused": {
        "weight_cost": 0.2,
        "weight_service": 0.7,
        "weight_investment": 0.1,
    },
    "Balanced": {
        "weight_cost": 0.33,
        "weight_service": 0.33,
        "weight_investment": 0.34,
    },
    "Investment-conscious": {
        "weight_cost": 0.3,
        "weight_service": 0.3,
        "weight_investment": 0.4,
    },
}

recommendation_df = pareto_df.copy()
recommendation_df["norm_cost"] = normalize_series(recommendation_df["total_cost"])
recommendation_df["norm_service_loss"] = 1 - normalize_series(
    recommendation_df["service_level"]
)
recommendation_df["norm_investment"] = normalize_series(
    recommendation_df["inventory_investment"]
)

recommendations = {}
print("DECISION-SUPPORT SCENARIOS")
print("=" * 60)
for scenario, weights in scenarios.items():
    weight_total = sum(weights.values())
    if not np.isclose(weight_total, 1.0):
        raise ValueError(f"Weights for {scenario} must sum to 1.")

    score_column = f"score_{scenario}"
    recommendation_df[score_column] = (
        weights["weight_cost"] * recommendation_df["norm_cost"]
        + weights["weight_service"] * recommendation_df["norm_service_loss"]
        + weights["weight_investment"] * recommendation_df["norm_investment"]
    )
    best_solution = recommendation_df.loc[
        recommendation_df[score_column].idxmin()
    ]
    recommendations[scenario] = best_solution

    print(f"\n{scenario} preference:")
    print(f"  Review period: {best_solution['review_period']:.3f} weeks")
    print(f"  Base-stock level: {best_solution['base_stock_level']:,.0f} units")
    print(f"  Total cost: ${best_solution['total_cost']:,.2f}/week")
    print(f"  Cycle service level: {best_solution['service_level'] * 100:.2f}%")
    print(f"  Inventory investment: ${best_solution['inventory_investment']:,.2f}")
    print(f"  Weighted normalized score: {best_solution[score_column]:.4f}")

print("\nInterpretation: each recommendation is conditional on its stated weights and the observed range of this approximate front.")

In [ ]:
print("=" * 70)
print("PERIODIC-REVIEW BASE-STOCK POLICY: TIER 3 SUMMARY")
print("=" * 70)
print("\nMethod")
print("- NSGA-II searches for a diverse approximation to the Pareto front.")
print("- The decision variables are review period R and base-stock level S.")
print("- The minimized objective vector is cost, negative service, inventory investment, and ordering frequency.")
print(
    f"- Population size: {ga_optimizer.pop_size}; "
    f"generations: {ga_optimizer.generations}."
)

print("\nObserved results from this run")
print(f"- Approximate Pareto-front candidates: {len(pareto_df)}")
print(f"- Weekly cost range: ${costs.min():,.2f} to ${costs.max():,.2f}")
print(f"- Cycle service range: {services.min() * 100:.2f}% to {services.max() * 100:.2f}%")
print(f"- Inventory investment range: ${investments.min():,.2f} to ${investments.max():,.2f}")
print("- The front is an approximate result from a stochastic, finite search.")

print("\nPreference-based selections")
for scenario, solution in recommendations.items():
    print(
        f"- {scenario}: R={solution['review_period']:.3f} weeks, "
        f"S={solution['base_stock_level']:,.0f} units, "
        f"cost=${solution['total_cost']:,.2f}/week, "
        f"service={solution['service_level'] * 100:.2f}%"
    )

print("\nModel interpretation")
print("- A Pareto-front policy is not dominated by another evaluated policy across all four objectives.")
print("- Preference weights support a conditional choice; they do not make one policy best for every decision maker.")
print("- The model assumes normal demand over the protection interval and does not include capacity or time-varying effects.")
print("- Results can vary between runs because the genetic search is stochastic.")